# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

The objective is to identify web pages experiencing organic traffic decay and rank them to output a prioritized review queue for content managers. While we output a continuous probability score to order the audit list (ranking/scoring), the underlying prediction target is binary (1 = declining page, 0 = stable or growing page).

In [10]:
import os
import sys
import subprocess
import pandas as pd
import numpy as np


REPO_NAME = "flyrank-ml-internship-starter"
REPO_URL = f"https://github.com/flyrank-bih/{REPO_NAME}.git"


if "google.colab" in sys.modules:
    if not os.path.exists(REPO_NAME):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    os.chdir(REPO_NAME)

while not os.path.exists("data/raw/content_refresh_anonymized.csv") and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir("..")

print(f"Current working directory: {os.getcwd()}")


DATA_PATH = "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)


df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(f"Successfully loaded dataset: {df.shape[0]} rows x {df.shape[1]} columns")

Current working directory: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Successfully loaded dataset: 30000 rows x 45 columns


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print("Task Type: Binary Classification & Priority Scoring")
print(f"Total dataset size: {len(df)} pages")
print("\nTarget Class Distribution:")
print(df["is_declining_label"].value_counts(normalize=True).rename({1: "Declining (1)", 0: "Stable/Growing (0)"}))

Task Type: Binary Classification & Priority Scoring
Total dataset size: 30000 pages

Target Class Distribution:
is_declining_label
Declining (1)         0.542067
Stable/Growing (0)    0.457933
Name: proportion, dtype: float64


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Predicting: is_declining_label (1 if trend_direction == 'down', else 0).

Label Origin: This target is an observed outcome derived directly from real search traffic performance over time (Google Search Console trend trajectory), rather than an arbitrary human-written heuristic or proxy flag.

In [4]:
# Derive binary target from observed search trend trajectory
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

declining_count = df["is_declining_label"].sum()
declining_rate = df["is_declining_label"].mean()

print(f"Observed Target Label: 'is_declining_label'")
print(f"Positive cases (declining pages): {declining_count} / {len(df)} ({declining_rate:.1%})")

Observed Target Label: 'is_declining_label'
Positive cases (declining pages): 16262 / 30000 (54.2%)


## 3. Success metric

*One metric you can defend. What number means 'good'?*

### **Primary Metric: Precision@K (specifically Precision@50)**

* **Defense:** Content auditing teams have a limited weekly capacity (e.g., manually auditing or rewriting ~50 pages per week). Precision@50 directly measures what percentage of the top 50 flagged recommendations are *actually* declining pages.
* **What means 'good':** A **Precision@50 $\ge$ 0.70 (70%)** means 7 out of 10 recommended pages are genuine decay targets, delivering a significant efficiency boost over random inspection (~54.2% baseline rate).

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Derive binary target from observed search trend trajectory
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

declining_count = df["is_declining_label"].sum()
declining_rate = df["is_declining_label"].mean()

print(f"Observed Target Label: 'is_declining_label'")
print(f"Positive cases (declining pages): {declining_count} / {len(df)} ({declining_rate:.1%})")

Observed Target Label: 'is_declining_label'
Positive cases (declining pages): 16262 / 30000 (54.2%)


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of Analysis: One row = One unique content page / URL slice evaluated over a performance observation window.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Select representative performance columns present in the dataset
sample_cols = [
    "content_id",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count",
    "is_declining_label"
]

print(f"Unit of analysis check: Dataframe has {df.shape[0]} rows (1 row = 1 unique content page).")
df[sample_cols].head(5)

Unit of analysis check: Dataframe has 30000 rows (1 row = 1 unique content page).


,content_id,days_since_last_update,impressions_90d,avg_position,ctr,word_count,is_declining_label
0,content_304f48230142,20,3803,10.6,0.76,3221.0,1
1,content_a1fb4e703a9e,25,15320,20.3,0.05,2481.0,1
2,content_9aa793d4d895,20,12581,36.5,0.09,3515.0,1
3,content_331d6c4de07b,22,11751,6.2,0.49,NaN,0
4,content_d99b7a2d90ca,14,19140,44.0,0.13,2803.0,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Why ML wins over an if-statement:

Coarse Thresholding & Heavy Rank Ties: A hardcoded rule (e.g., days_since_last_update > 180 AND impressions > 500) creates sharp cutoffs and produces identical tie scores across top candidate pages, making fine-grained prioritization impossible.

Multi-Signal Interactions: Traffic decay involves complex interactions across continuous metrics—such as rank drift (avg_position), CTR shifts (ctr), traffic volume (impressions_90d), and page age (days_since_last_update). ML models combine these signals into continuous probability scores that accurately rank decay risk.

In [9]:
import numpy as np

def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

top_50_hand_rule = df["hand_rule_score"].nlargest(50)

print(f"Unique score values in Top 50 hand-rule results: {top_50_hand_rule.nunique()} (Heavy ties limit effective ranking)")
print(f"Hand Rule Precision@50: {precision_at_k(df['hand_rule_score'], df['is_declining_label'], 50):.3f}")

Unique score values in Top 50 hand-rule results: 18 (Heavy ties limit effective ranking)
Hand Rule Precision@50: 0.680


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.